# 03 - Model Training

## Notebook Objectives
- Train a forecasting model on the engineered features to learn normal daily demand behavior.
- Load the prepared train and test artifacts from the project pipeline to keep notebook and script workflows consistent.
- Compare an untuned LightGBM baseline with a tuned LightGBM model to verify that tuning improves performance.
- Evaluate model quality on holdout test data using MAE and MAPE.
- Save the selected model as the reusable reference model for downstream anomaly injection and detection notebooks.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root))

from config import MODEL_TEST_DATA, MODEL_TRAIN_DATA, TUNED_MODEL_PATH
from src.config_loader import load_config
from src.features.utils import get_feature_columns
from src.models.io import save_model
from src.models.lgbm_tuner import tune_lgbm_model

## Load prepared model artifacts

Train/test datasets are prepared by the script workflow (`prepare_model_data.py`) and persisted as role-based artifacts. This notebook consumes those artifacts directly rather than re-splitting data.

In [2]:
cfg = load_config()
time_col = cfg["time"]["name"]
target_col = cfg["target"]["name"]

train_df = pd.read_csv(MODEL_TRAIN_DATA, parse_dates=[time_col])
test_df = pd.read_csv(MODEL_TEST_DATA, parse_dates=[time_col])

print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
# print(f"Loaded: {MODEL_TRAIN_DATA}")
# print(f"Loaded: {MODEL_TEST_DATA}")

Train rows: 144
Test rows: 29


## Train and tune the model

We tune a LightGBM regressor to fit the normal demand pattern as closely as possible. We also fit a simple, untuned LightGBM as a baseline, just to confirm the tuning effort actually pays off. The tuned model becomes our reference for "expected" behavior when we test it against injected anomalies in notebook 04.

In [3]:
feature_cols = get_feature_columns()

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

# Simple untuned baseline to check tuning is worth the effort
baseline_model = LGBMRegressor(random_state=42, verbose=-1)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_mape = mean_absolute_percentage_error(y_test, baseline_pred)

# Hyperparameter tuning
model_tuned, best_params, best_cv_mae, cv_results = tune_lgbm_model(
    X_train,
    y_train,
    random_state=42,
    n_splits=4,
    n_iter=10,
)

# Inspect CV performance of the best tuned model
best_idx = cv_results["rank_test_score"].argmin()

fold_maes = [
    -cv_results[f"split{i}_test_score"][best_idx]
    for i in range(4)
]

print("\nBest tuned model CV results:")
for i, mae in enumerate(fold_maes, start=1):
    print(f"  Fold {i}: MAE = {mae:.2f}")

print(f"  Mean: {best_cv_mae:.2f}")
print(f"  Std:  {cv_results['std_test_score'][best_idx]:.2f}")

# Evaluate tuned model on the held-out test set
y_pred = model_tuned.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred)
test_mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"\nBaseline Test MAE: {baseline_mae:.2f} | Baseline Test MAPE: {baseline_mape:.4f}")
print(f"Tuned CV MAE: {best_cv_mae:.2f}")
print(f"Tuned Test MAE: {test_mae:.2f} | Tuned Test MAPE: {test_mape:.4f}")

mae_change_pct = (baseline_mae - test_mae) / baseline_mae * 100
print(f"Tuned model vs. baseline changes MAE by {mae_change_pct:.1f}%")

print("\nBest params:")
for k, v in sorted(best_params.items()):
    print(f"  {k}: {v}")

Fitting 4 folds for each of 10 candidates, totalling 40 fits

Best tuned model CV results:
  Fold 1: MAE = 19835.08
  Fold 2: MAE = 5391.83
  Fold 3: MAE = 4544.64
  Fold 4: MAE = 3829.81
  Mean: 8400.34
  Std:  6624.96

Baseline Test MAE: 4106.44 | Baseline Test MAPE: 0.0350
Tuned CV MAE: 8400.34
Tuned Test MAE: 3732.71 | Tuned Test MAPE: 0.0324
Tuned model vs. baseline changes MAE by 9.1%

Best params:
  colsample_bytree: 1.0
  learning_rate: 0.03
  max_depth: 12
  min_child_samples: 10
  n_estimators: 200
  num_leaves: 31
  reg_alpha: 0.1
  reg_lambda: 1.0
  subsample: 1.0


## Model Interpretation and Selection

- Hyperparameter tuning improved both cross-validation and hold-out test performance.
- The tuned model reduced test MAE from 4,106 to 3,733 (9.1%) and test MAPE from 3.50% to 3.24%.
- Therefore, the tuned model is selected as the forecasting model for the subsequent anomaly detection pipeline.
- Given the small number of observations and substantial variation across the time-series folds, the improvement should be interpreted cautiously.

## Save the selected model

We persist the tuned model to disk, so the next notebook can load it directly instead of retraining.

In [4]:
save_model(model_tuned, TUNED_MODEL_PATH)
# print(f"Saved tuned model to: {TUNED_MODEL_PATH}")
